# Deep Learning for Genomics (Review) · Methodology

This notebook documents the **methodology** used for the review-paper assignment:
**Eraslan, Avsec, Gagneur & Theis, *Deep learning: new computational modelling techniques
for genomics*, Nature Reviews Genetics 20, 389–403 (2019).**

Unlike the other four papers, this one **surveys the field** rather than shipping a single tool,
so there is no author codebase to run. The methodology adopted here is therefore:

> **Demonstrate one representative deep-learning method end to end** — from sequence encoding to
> model interpretation — and connect it to the team's **variant-prioritisation** goal.

The runnable pipeline lives in [`deep_learning_genomics_review_demo.ipynb`](./deep_learning_genomics_review_demo.ipynb);
this notebook explains the reasoning behind each step and how it reflects the concepts in the review.

## 1. What the review covers, and what we reproduce

The review organises deep learning in genomics around a few recurring ideas:

- **Sequence as input** — DNA/RNA is one-hot encoded so convolutional networks can read it.
- **Convolutional neural networks (CNNs)** — learn motifs (e.g. transcription-factor binding sites)
  as convolution filters; used by DeepBind, DeepSEA, Basset, and (paper 04) Akita.
- **Recurrent networks, multitask & transfer learning** — for longer-range or shared signals.
- **Model interpretation** — figuring out *what the network learned*, e.g. via
  **in-silico saturation mutagenesis (ISM)**, which scores the effect of every possible variant.
- **Generative / unsupervised models** — autoencoders and GANs.

We cannot reproduce all of these, so we implement the **representative supervised path** the review
treats as the field's workhorse: **one-hot DNA → CNN motif detector → interpretation via ISM →
variant ranking.** This is the same interpretation technique Akita applies to Hi-C maps, which ties
paper 05 directly to paper 04.

## 2. Data — a controlled motif-detection task

To keep the demo **fully self-contained** (no downloads, runs in seconds on CPU), we generate
**synthetic** sequences with a known ground truth:

- **Positive** sequences contain a planted **CTCF-like motif** (`CCGCGNGGNGGCAG`, where `N` is a
  random base) at a random position.
- **Negative** sequences are random DNA of the same length.

Because *we* placed the motif, we know exactly where the signal is — which lets us later check
whether the model's interpretation recovers it. Each sequence is **one-hot encoded** to `[4, L]`,
the standard representation described in the review.

In [ ]:
# From deep_learning_genomics_review_demo.ipynb — encode + build the labelled dataset
BASES = "ACGT"; IDX = {b:i for i,b in enumerate(BASES)}
def one_hot(s):
    m = np.zeros((4,len(s)),np.float32)
    for j,b in enumerate(s): m[IDX[b],j]=1
    return m

SEQ_LEN, MOTIF, N = 200, "CCGCGNGGNGGCAG", 2000
rand = lambda L: "".join(np.random.choice(list(BASES),L))
inst = lambda m: "".join(np.random.choice(list(BASES)) if b=="N" else b for b in m)
# 2000 positives (motif planted) + 2000 negatives (random) -> one-hot X, labels y

## 3. Model — a minimal DeepBind / Basset-style CNN

The architecture mirrors the canonical sequence CNN from the review:

1. **1D convolution** (`Conv1d(4, 16, kernel_size=15)`) — the 16 filters act as learnable
   **motif detectors** scanning the sequence; kernel width 15 roughly matches the motif length.
2. **Global max-pooling** (`.max(2)`) — records the *strongest* activation of each filter anywhere
   in the sequence. This gives **position invariance**: the motif is detected wherever it occurs.
3. **Linear head** → a single **binding/no-binding** logit.

It is deliberately tiny (a few hundred parameters) so it trains in seconds while still exhibiting
the behaviour the review describes.

In [ ]:
class CNN(nn.Module):
    def __init__(s):
        super().__init__(); s.c=nn.Conv1d(4,16,15); s.f=nn.Linear(16,1)
    def forward(s,x):
        return s.f(torch.relu(s.c(x)).max(2).values)   # conv -> ReLU -> global max-pool -> linear

## 4. Training

Standard supervised setup for a binary classifier:

- **Loss:** binary cross-entropy (`BCEWithLogitsLoss`).
- **Optimiser:** Adam, learning rate 1e-3.
- **Schedule:** 15 epochs, minibatches of 128, on an 80/20 train/test split.

The training loss falls from ≈0.69 (random guessing) toward ≈0.16, confirming the filters are
learning the planted motif.

In [ ]:
model = CNN(); opt = torch.optim.Adam(model.parameters(), 1e-3); lf = nn.BCEWithLogitsLoss()
for e in range(15):
    o = torch.randperm(len(Xt));
    for i in range(0, len(Xt), 128):
        j = o[i:i+128]; opt.zero_grad(); L = lf(model(Xt[j]), yt[j]); L.backward(); opt.step()

## 5. Evaluation

On the held-out test set we report **accuracy** and **ROC-AUC**. Because the task is clean and the
signal is real, the model reaches near-perfect scores (**accuracy ≈ 0.99, ROC-AUC ≈ 1.00**) — the
point is not the headline number but that the model has genuinely located the motif, which we verify
next.

In [ ]:
with torch.no_grad(): p = torch.sigmoid(model(torch.tensor(Xte))).numpy().ravel()
acc = float(((p > .5) == yte).mean())
auc = roc_auc_score(yte, p)
print("accuracy:", round(acc, 3), "| ROC-AUC:", round(auc, 3))

## 6. Interpretation — in-silico saturation mutagenesis (ISM)

This is the methodological centrepiece and the direct link to variant prioritisation. **ISM** asks:
*how does the prediction change if we mutate each position to each alternative base?*

1. Take a positive sequence and record the model's reference probability `P_ref`.
2. For **every position** and **every alternative base**, build the mutated one-hot sequence and
   recompute the probability.
3. The **effect score** of a variant is `P_mut − P_ref`.

Positions where mutations sharply *reduce* the score are exactly the positions the model relies on —
i.e. the planted motif. The model **localises the motif on its own**, without ever being told where
it is.

In [ ]:
seq = Xte[np.where(yte == 1)[0][0]]; ref = "".join(BASES[i] for i in seq.argmax(0))
def prob(m):
    with torch.no_grad(): return torch.sigmoid(model(torch.tensor(m[None]))).item()
r0 = prob(one_hot(ref)); D = np.zeros((4, SEQ_LEN), np.float32)
for j in range(SEQ_LEN):                       # every position
    for b in range(4):                         # every alternative base
        mut = one_hot(ref).copy(); mut[:, j] = 0; mut[b, j] = 1
        D[b, j] = prob(mut) - r0               # variant effect = P(mut) - P(ref)

The effect matrix `D` is visualised as a **variant-effect heatmap** (position × alternative base),
and the largest-magnitude entries are the **highest-impact single-nucleotide variants**. Ranking the
variants by `|Δ score|` produces a prioritised shortlist:

In [ ]:
ri = one_hot(ref).argmax(0)
df = pd.DataFrame([{"pos": j, "ref": BASES[ri[j]], "alt": BASES[b], "dscore": float(D[b, j])}
                   for j in range(SEQ_LEN) for b in range(4) if b != ri[j]])
df["abs"] = df["dscore"].abs()
top10 = df.sort_values("abs", ascending=False).drop(columns="abs").head(10).reset_index(drop=True)
top10   # the 10 highest-impact single-nucleotide variants

## 7. Connection to variant prioritisation

This mini-pipeline is a scaled-down version of the whole team's objective:

- A **sequence model** learns what "functional" DNA looks like.
- **ISM interpretation** turns that model into a **variant scorer** — every possible mutation gets a
  predicted effect.
- **Ranking by effect size** yields a **prioritised list of variants**.

The exact same logic scales up to the real tools in the review and in this repo: DeepSEA/Basset score
regulatory variants, and **Akita (paper 04)** scores variants by how much they distort predicted 3D
folding. Paper 05 therefore provides the **conceptual backbone** — *encode → learn → interpret → rank* —
that the other papers instantiate on real data.

## 8. Summary of the methodology

| Step | What happens | Concept from the review |
|------|--------------|-------------------------|
| 1 | One-hot encode DNA | Sequence as input |
| 2 | Synthetic motif dataset | Controlled ground truth |
| 3 | Conv → max-pool → linear | CNN motif detector (DeepBind/Basset) |
| 4 | Train with BCE + Adam | Supervised learning |
| 5 | Accuracy / ROC-AUC | Model evaluation |
| 6 | Saturation mutagenesis | Model interpretation (ISM) |
| 7 | Rank variants by \|Δ score\| | Variant prioritisation |

Everything uses **synthetic data with a fixed seed**, so this is an *illustrative reproduction of the
workflow* the review describes — deliberately not the real Akita model (see
[`../04-akita/`](../04-akita/) for inference with the actual pre-trained network). It runs end to end
in seconds on CPU and demonstrates the full encode → learn → interpret → prioritise loop.